In [3]:
import glob
import pickle
from tqdm import tqdm
import pandas as pd
import h5py
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import re
import pandas as pd

In [4]:
OG_DATASET = "/home/mdelafuente/pipeline/pipeline/training/lc_classifier_ztf/ATAT_ALeRCE/data/datasets/ZTF_ff/final/LC_MD_FEAT_240627_windows_200_12/dataset.h5"


In [5]:
n = 1000
these_idx = None
with h5py.File(OG_DATASET, 'r') as f_:
    #these_idx = f_.get('training_0')[:n]
    these_idx = np.random.randint(0, len(f_.get('training_0')), size = (n,))
    #these_idx.sort()
    
    train_0_labels = [f_.get('labels')[i].astype(int) for i in these_idx]
    train_0_data = [f_.get('flux')[i] for i in these_idx]
    train_0_time = [f_.get('time')[i] for i in these_idx]
    these_idx = [f_.get('training_0')[i].astype(int) for i in range(these_idx.shape[0])]

In [6]:
train_0_labels = np.array(train_0_labels)
train_0_data = np.array(train_0_data)
train_0_time = np.array(train_0_time)
count, bins = np.histogram(train_0_labels, bins=np.arange(22))
print(count.shape, bins.shape)
#plt.bar(bins[:-1], count)
#plt.xticks(np.arange(0,22))

(21,) (22,)


In [ ]:
from torch.utils.data import Sampler, BatchSampler

class FunctionBatchSampler(BatchSampler):
    def __init__(self, sampler, batch_size, drop_last, include_fn):
        """
        sampler: base sampler (e.g., SequentialSampler, RandomSampler)
        batch_size: number of elements per batch
        drop_last: drop the last incomplete batch
        include_fn: function(index) -> bool
        """
        self.sampler = sampler
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.include_fn = include_fn

    def __iter__(self):
        batch = []
        for idx in self.sampler:
            if self.include_fn(idx):
                batch.append(idx)
                if len(batch) == self.batch_size:
                    yield batch
                    batch = []
        if batch and not self.drop_last:
            yield batch

    def __len__(self):
        # Length estimation can be tricky since it depends on include_fn.
        # You might want to pass a precomputed length or make this dynamic.
        return len(self.sampler) // self.batch_size
